# Spatial filters from naturalistic stimuli

LPD on translated 1D dead-leaves stimuli (Section 4, Fig. 3b left).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

RESULTS_DIR = "results"  # output folder for figures
DATA_DIR = "data"  # output folder for intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Stimulus and past/future increments

In [ ]:
T = 10000
t = np.arange(1, T + 1)

# 1D dead-leaves signal: piecewise-constant plateaus with smooth edges
plateau_vals = np.array(
    [
        0, 1, -0.5, 1.5, 0.3, 1.1, -0.7, 1.2, -0.2, 0.1, 1.3, -0.75, 1.25, 0.33, 1.21,
        -0.78, 1.22, -0.125, 0.04, 1.16, -0.52, 1.55, 0.36, -1.15, -0.73, 1.29, -0.22,
        0.17, -1.3, -0.55, -1.25, 0.43, 1.11, -0.98, 1.75, -0.9
    ]
)
num_transitions = plateau_vals.size - 1

transition_centers = np.linspace(T / num_transitions, T, num_transitions + 1)
transition_centers = transition_centers[1:]
transition_width = 5.0

x = plateau_vals[0] * np.ones(T)
for k in range(num_transitions):
    v_left = plateau_vals[k]
    v_right = plateau_vals[k + 1]
    c_k = transition_centers[k]

    s_k = 0.5 * (1 + np.tanh((t - c_k) / transition_width))
    x = x + (v_right - v_left) * s_k

a = x + 1e-6 * np.random.randn(T)

margin = 55
length = 50
X = []
Y = []
# Past/future temporal differences of windows translated in both directions
for start_y in range(margin, a.shape[0] - margin):

    x1 = a[start_y - length : start_y + length - 1].flatten()
    x2 = a[start_y - length + 1 : start_y + length + 1 - 1].flatten()
    x3 = a[start_y - length + 2 : start_y + length + 2 - 1].flatten()
    x4 = a[start_y - length + 3 : start_y + length + 3 - 1].flatten()
    x = x2 - x1
    y = x4 - x3
    x += 1e-6 * np.random.randn(x.shape[0])
    y += 1e-6 * np.random.randn(y.shape[0])

    X.append(x)
    Y.append(y)

    x1 = a[start_y - length : start_y + length - 1].flatten()
    x2 = a[start_y - length - 1 : start_y + length - 1 - 1].flatten()
    x3 = a[start_y - length - 2 : start_y + length - 2 - 1].flatten()
    x4 = a[start_y - length - 3 : start_y + length - 3 - 1].flatten()
    x = x2 - x1
    y = x4 - x3
    x += 1e-6 * np.random.randn(x.shape[0])
    y += 1e-6 * np.random.randn(y.shape[0])

    X.append(x)
    Y.append(y)

X = np.array(X)
Y = np.array(Y)

X = torch.from_numpy(X).to(device)
Y = torch.from_numpy(Y).to(device)

Cxx = X.T @ X / T / 2
Cxx += 2e-4 * torch.eye(Cxx.shape[0]).to(device)
Cxy = X.T @ Y / T / 2
Cxy += 2e-4 * torch.eye(Cxy.shape[0]).to(device)
Cyy = Y.T @ Y / T / 2
Cyy += 2e-4 * torch.eye(Cyy.shape[0]).to(device)

## LPD

In [ ]:
Nz_max = 100
a_z = 0.1
a_W = 0.2
a_M = 0.2

T, n = X.shape
m = 20
rf = torch.randn(m, n).to(device)
error = 10.0
count = 0


def orthogonal_matrix(dim):
    Q, _ = np.linalg.qr(np.random.randn(dim, dim))
    return Q

M = np.eye(m)
W = orthogonal_matrix(m) @ np.eye(m, n) @ orthogonal_matrix(n) * 0.1
M = torch.from_numpy(M).to(device)
W = torch.from_numpy(W).to(device)
Z = torch.rand(T, m).to(device).double()
Z[Z < 0.0] = 0.0
Z = Z / torch.linalg.norm(Z, axis=0, keepdims=True)

# Offline LPD with rectified inference
with torch.no_grad():
    while count <= 3000:

        for _ in range(Nz_max):
            Z_new = (W @ X.T - M @ Z.T).T
            Z = a_z * Z_new + (1 - a_z) * Z
            Z[Z < 0.0] = 0.0
        Z = Z / torch.linalg.norm(Z, axis=0, keepdims=True)

        M_new = Z.T @ Z
        Czx = Z.T @ X / np.sqrt(T)
        W_new = (
            Czx
            @ torch.linalg.inv(Cxx)
            @ Cxy
            @ torch.linalg.inv(Cyy)
            @ Cxy.T
            @ torch.linalg.inv(Cxx)
        )
        M = a_M * M_new + (1 - a_M) * M
        W = a_W * W_new + (1 - a_W) * W

        error = torch.abs(rf - torch.linalg.inv(M) @ W).sum() / torch.abs(rf).sum()
        rf = torch.linalg.inv(M) @ W
        count += 1
        if count % 300 == 1:
            print(error)

In [ ]:
cmap = plt.colormaps["tab20"]
colors = cmap(np.linspace(0, 1, 20))
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(20):
    if i == 7:
        ax.plot(rf.detach().cpu()[i], color="magenta", linewidth=5)
    elif rf.detach().cpu()[i].sum() > 0:
        ax.plot(rf.detach().cpu()[i], color=colors[i], alpha=0.8)
    else:
        ax.plot(rf.detach().cpu()[i], color=colors[i], alpha=0.8, linestyle="dashed")
ax.set_ylabel("Spatial filter", fontsize=15)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.xaxis.set_major_locator(plt.MaxNLocator(4))
ax.yaxis.set_major_locator(plt.MaxNLocator(4))
ax.tick_params(axis="both", which="major", labelsize=13)
ax.set_xlabel("Location (pixel)", fontsize=15)
ax.set_ylim([-4, 4.3])
ax.set_yticks([-4, 0, 4])

plt.savefig(
    os.path.join(RESULTS_DIR, "spatial_filters.png"), dpi=600, bbox_inches="tight"
)
plt.show()